# Step 2 — Data Preparation

**LB2 Project · Group 7 · Signal Peptide Prediction**

### Objective
Starting from the raw positive and negative datasets retrieved in Step 1, this notebook:
1. Removes redundancy using **MMseqs2** clustering
2. Filters metadata to keep only cluster representatives
3. Adds binary labels (`1` = has signal peptide, `0` = no signal peptide)
4. Performs a **stratified 80/20 train/benchmark split**
5. Assigns **5-fold cross-validation** labels to the training set

All outputs are ready for deep learning model training in Step 6.

### Current Dataset (as of this run)

| Stage                        | Positive | Negative | Total  |
|------------------------------|----------|----------|--------|
| Raw (from Step 1)            | 2,932    | 20,615   | 23,547 |
| After MMseqs2 clustering     | **1,093**| **8,934**| 10,027 |
| Training set (80 %)          | 874      | 7,147    | **8,021** |
| Benchmarking set (20 %)      | 219      | 1,787    | **2,006** |

**Note:** All random operations use `RANDOM_SEED = 42` for full reproducibility.

---
## Cell 1 — Install dependencies

In [ ]:
# MMseqs2 for sequence clustering
!apt-get install -y -q mmseqs2

# Python libraries
!pip install -q biopython pandas numpy

print('Done.')

---
## Cell 2 — Imports and global settings

This cell imports required libraries and sets the fixed random seed used throughout the entire pipeline for reproducibility.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from Bio import SeqIO

# Fixed random seed — do NOT change this.
# Every shuffle and split uses this seed so the entire pipeline is reproducible.
RANDOM_SEED = 42

# Directories for MMseqs2 output and temp files
os.makedirs('cluster_output', exist_ok=True)
os.makedirs('mmseqs_tmp',    exist_ok=True)

def find_mmseqs_rep_fasta(prefix):
    """Return the representative FASTA produced by MMseqs2 for a given prefix.

    Tries the expected filename first, then falls back to a pattern search.
    Raises a clear FileNotFoundError if nothing suitable is found.
    """
    expected = f"{prefix}_rep_seq.fasta"
    if os.path.exists(expected):
        return expected

    parent = os.path.dirname(prefix) or "."
    stem = os.path.basename(prefix)

    candidates = sorted(
        p for p in glob.glob(os.path.join(parent, f"{stem}*"))
        if p.endswith(".fasta")
    )

    # Prefer representative-like names if present
    preferred = [p for p in candidates if "rep" in os.path.basename(p).lower()]
    if preferred:
        return preferred[0]
    if candidates:
        return candidates[0]

    available = sorted(os.listdir(parent)) if os.path.isdir(parent) else []
    raise FileNotFoundError(
        f"No FASTA output found for MMseqs prefix '{prefix}'.\n"
        f"Expected: {expected}\n"
        f"Available files in {parent}: {available}"
    )

print('Imports OK.')
print('Random seed:', RANDOM_SEED)

---
## Cell 3 — Verify raw input files

In [ ]:
# Count FASTA entries
n_pos_fasta = sum(1 for _ in SeqIO.parse('positive.fasta', 'fasta'))
n_neg_fasta = sum(1 for _ in SeqIO.parse('negative.fasta', 'fasta'))

# Load TSV files
df_pos_raw = pd.read_csv('positive.tsv', sep='\t')
df_neg_raw = pd.read_csv('negative.tsv', sep='\t')

print(f'positive.fasta : {n_pos_fasta:,} sequences')
print(f'negative.fasta : {n_neg_fasta:,} sequences')
print(f'positive.tsv   : {len(df_pos_raw):,} rows | columns: {df_pos_raw.columns.tolist()}')
print(f'negative.tsv   : {len(df_neg_raw):,} rows | columns: {df_neg_raw.columns.tolist()}')

# Sanity checks
assert n_pos_fasta == len(df_pos_raw), \
    f'Mismatch: {n_pos_fasta} FASTA sequences vs {len(df_pos_raw)} TSV rows (positive)'
assert n_neg_fasta == len(df_neg_raw), \
    f'Mismatch: {n_neg_fasta} FASTA sequences vs {len(df_neg_raw)} TSV rows (negative)'

overlap_raw = set(df_pos_raw['Accession']) & set(df_neg_raw['Accession'])
assert len(overlap_raw) == 0, f'{len(overlap_raw)} accessions appear in both pos and neg!'

print()
print('All input checks passed. ✓')

---
## Cell 4 — Cluster positive sequences with MMseqs2

**Why clustering?** Without it, highly similar proteins from the same family would appear in both
training and validation folds, inflating performance. We cluster at 30% identity / 40% coverage
and keep one representative per cluster.

**MMseqs2 parameters:**

| Parameter | Value | Meaning |
|---|---|---|
| `--min-seq-id` | 0.3 | Minimum 30% sequence identity within a cluster |
| `-c` | 0.4 | Minimum 40% coverage |
| `--cov-mode` | 0 | Coverage over both query and target |
| `--cluster-mode` | 1 | Connected-component clustering |

In [ ]:
# Run MMseqs2 clustering for positive sequences
!mmseqs easy-cluster positive.fasta \
    cluster_output/pos \
    mmseqs_tmp \
    --min-seq-id 0.3 \
    -c 0.4 \
    --cov-mode 0 \
    --cluster-mode 1 \
    -v 1

print("\nFiles currently in cluster_output:")
print(sorted(os.listdir("cluster_output")))

pos_rep_file = find_mmseqs_rep_fasta("cluster_output/pos")
print(f"\nUsing representative FASTA: {pos_rep_file}")

pos_reps = list(SeqIO.parse(pos_rep_file, "fasta"))

print(f'\nPositive — before clustering : {n_pos_fasta:,}')
print(f'Positive — after clustering  : {len(pos_reps):,} (representatives)')
print(f'Redundancy removed           : {100*(1 - len(pos_reps)/n_pos_fasta):.1f}%')

---
## Cell 5 — Cluster negative sequences with MMseqs2

In [ ]:
# Run MMseqs2 clustering for negative sequences
!mmseqs easy-cluster negative.fasta \
    cluster_output/neg \
    mmseqs_tmp \
    --min-seq-id 0.3 \
    -c 0.4 \
    --cov-mode 0 \
    --cluster-mode 1 \
    -v 1

print("\nFiles currently in cluster_output:")
print(sorted(os.listdir("cluster_output")))

neg_rep_file = find_mmseqs_rep_fasta("cluster_output/neg")
print(f"\nUsing representative FASTA: {neg_rep_file}")

neg_reps = list(SeqIO.parse(neg_rep_file, "fasta"))

print(f'\nNegative — before clustering : {n_neg_fasta:,}')
print(f'Negative — after clustering  : {len(neg_reps):,} (representatives)')
print(f'Redundancy removed           : {100*(1 - len(neg_reps)/n_neg_fasta):.1f}%')

---
## Cell 6 — Filter TSV metadata to cluster representatives

In [ ]:
# Collect representative accession IDs from FASTA headers
pos_rep_ids = set(rec.id for rec in pos_reps)
neg_rep_ids = set(rec.id for rec in neg_reps)

# No accession should appear in both sets
assert len(pos_rep_ids & neg_rep_ids) == 0, 'Overlap in representative IDs!'

# Filter TSV rows to keep only representatives
df_pos = df_pos_raw[df_pos_raw['Accession'].isin(pos_rep_ids)].copy().reset_index(drop=True)
df_neg = df_neg_raw[df_neg_raw['Accession'].isin(neg_rep_ids)].copy().reset_index(drop=True)

# Every representative must be accounted for in the TSV
assert len(df_pos) == len(pos_rep_ids), \
    f'Only {len(df_pos)} of {len(pos_rep_ids)} positive reps found in TSV'
assert len(df_neg) == len(neg_rep_ids), \
    f'Only {len(df_neg)} of {len(neg_rep_ids)} negative reps found in TSV'

# Save
df_pos.to_csv('filtered_positive.tsv', sep='\t', index=False)
df_neg.to_csv('filtered_negative.tsv', sep='\t', index=False)

print(f'Filtered positive : {len(df_pos):,} rows → filtered_positive.tsv')
print(f'Filtered negative : {len(df_neg):,} rows → filtered_negative.tsv')
print('All assertions passed. ✓')

---
## Cell 7 — Add binary labels and merge

In [ ]:
df_pos['label'] = 1  # has signal peptide
df_neg['label'] = 0  # no signal peptide

# Merge into one labeled DataFrame
df_all = pd.concat([df_pos, df_neg], ignore_index=True)

n_pos_total = (df_all['label'] == 1).sum()
n_neg_total = (df_all['label'] == 0).sum()

print(f'Total labeled sequences : {len(df_all):,}')
print(f'  Positive (label=1)    : {n_pos_total:,}')
print(f'  Negative (label=0)    : {n_neg_total:,}')
print(f'  Imbalance ratio       : {n_neg_total/n_pos_total:.1f} negatives per positive')

---
## Cell 8 — Stratified 80/20 train/benchmark split

The split is performed **within each class separately** to preserve the positive/negative ratio. The benchmarking set is held out completely and used only once at the final evaluation.

In [ ]:
def stratified_split(df, train_frac=0.8, seed=42):
    """
    Split df into (df_train, df_bench) stratified by the 'label' column.

    Args:
        df         : labeled DataFrame
        train_frac : fraction to assign to training (default 0.8)
        seed       : random seed

    Returns:
        df_train, df_bench — both with reset indices, zero overlap guaranteed
    """
    rng = np.random.default_rng(seed)
    train_indices, bench_indices = [], []

    for label_val in sorted(df['label'].unique()):
        idx = df[df['label'] == label_val].index.to_numpy()
        rng.shuffle(idx)
        cut = int(len(idx) * train_frac)
        train_indices.extend(idx[:cut])
        bench_indices.extend(idx[cut:])

    df_train = df.loc[train_indices].reset_index(drop=True)
    df_bench = df.loc[bench_indices].reset_index(drop=True)
    return df_train, df_bench


df_train, df_bench = stratified_split(df_all, train_frac=0.8, seed=RANDOM_SEED)

# Verify integrity
assert len(set(df_train['Accession']) & set(df_bench['Accession'])) == 0, \
    'Overlap between train and benchmark!'
assert len(df_train) + len(df_bench) == len(df_all), \
    'Sequences lost during split!'

print('Training set:')
print(f'  Total    : {len(df_train):,}')
print(f'  Positive : {(df_train["label"]==1).sum():,}')
print(f'  Negative : {(df_train["label"]==0).sum():,}')
print()
print('Benchmarking set:')
print(f'  Total    : {len(df_bench):,}')
print(f'  Positive : {(df_bench["label"]==1).sum():,}')
print(f'  Negative : {(df_bench["label"]==0).sum():,}')
print()
print('Zero overlap between train and benchmark. ✓')
print('No sequences lost. ✓')

---
## Cell 9 — 5-fold cross-validation assignment

Each training sequence receives a fold number (0–4), stratified by label. In each CV round, fold `k` is used as the validation set.

In [ ]:
def assign_cv_folds(df, n_folds=5, seed=42):
    """
    Add a 'fold' column (0..n_folds-1) to df, stratified by 'label'.
    np.array_split distributes remainder rows to the last folds automatically.

    Args:
        df      : training DataFrame with a 'label' column
        n_folds : number of folds (default 5)
        seed    : random seed

    Returns:
        df with an added 'fold' column
    """
    rng = np.random.default_rng(seed)
    df = df.copy()
    df['fold'] = -1  # sentinel to detect unassigned rows

    for label_val in sorted(df['label'].unique()):
        idx = df[df['label'] == label_val].index.to_numpy()
        rng.shuffle(idx)
        # array_split handles uneven sizes — later chunks may have 1 extra row
        for fold_num, chunk in enumerate(np.array_split(idx, n_folds)):
            df.loc[chunk, 'fold'] = fold_num

    unassigned = (df['fold'] == -1).sum()
    assert unassigned == 0, f'{unassigned} rows not assigned to any fold!'
    return df


df_train = assign_cv_folds(df_train, n_folds=5, seed=RANDOM_SEED)

print(f'{"Fold":<8} {"Total":<10} {"Positive":<12} {"Negative"}')
print('-' * 42)
for fold in range(5):
    fdf = df_train[df_train['fold'] == fold]
    pos = (fdf['label'] == 1).sum()
    neg = (fdf['label'] == 0).sum()
    print(f'{fold:<8} {len(fdf):<10} {pos:<12} {neg}')

print()
print(f'Total training rows : {len(df_train):,}')
print('All folds assigned. ✓')

---
## Cell 10 — Save all outputs

In [ ]:
df_train.to_csv('training_with_folds.tsv', sep='\t', index=False)
df_bench.to_csv('benchmarking_set.tsv',    sep='\t', index=False)

print('Saved:')
print(f'  filtered_positive.tsv   {len(df_pos):>6,} rows  {df_pos.columns.tolist()}')
print(f'  filtered_negative.tsv   {len(df_neg):>6,} rows  {df_neg.columns.tolist()}')
print(f'  training_with_folds.tsv {len(df_train):>6,} rows  {df_train.columns.tolist()}')
print(f'  benchmarking_set.tsv    {len(df_bench):>6,} rows  {df_bench.columns.tolist()}')

---
## Final Summary

**Data Preparation Complete — LB2 Group 7**

- **Clustering tool**: MMseqs2 (`--min-seq-id 0.3 -c 0.4 --cov-mode 0 --cluster-mode 1`)
- **Splitting strategy**: Stratified by label (80 % train / 20 % benchmark)
- **Cross-validation**: 5-fold, stratified by label
- **Random seed**: 42 (fixed for reproducibility)

**Output files generated:**
- `filtered_positive.tsv` (1,093 entries)
- `filtered_negative.tsv` (8,934 entries)
- `training_with_folds.tsv` (8,021 entries + fold column)
- `benchmarking_set.tsv` (2,006 entries — **held-out**)

**Next step:** Proceed to the deep learning model training notebook (`step6_deep_learning_latest_version_negin.ipynb`).

In [ ]:
sep = '=' * 58
print(sep)
print('  DATA PREPARATION SUMMARY — LB2 Group 7')
print(sep)
print()
print(f'  {"Step":<40} {"Positive":>7} {"Negative":>7}')
print('  ' + '-' * 54)
print(f'  {"Raw sequences (UniProtKB)":<40} {n_pos_fasta:>7,} {n_neg_fasta:>7,}')
print(f'  {"After MMseqs2 clustering":<40} {len(df_pos):>7,} {len(df_neg):>7,}')
print(f'  {"Training set (80%)":<40} {(df_train["label"]==1).sum():>7,} {(df_train["label"]==0).sum():>7,}')
print(f'  {"Benchmarking set (20%)":<40} {(df_bench["label"]==1).sum():>7,} {(df_bench["label"]==0).sum():>7,}')
print()
print('  Clustering')
print('    Tool         : MMseqs2 easy-cluster')
print('    Identity     : 30%  (--min-seq-id 0.3)')
print('    Coverage     : 40%  (-c 0.4 --cov-mode 0)')
print('    Cluster mode : 1 (connected component)')
print()
print('  Splitting')
print('    Strategy     : Stratified by label (pos/neg ratio preserved)')
print('    Train/bench  : 80% / 20%')
print('    CV folds     : 5 (stratified)')
print(f'    Random seed  : {RANDOM_SEED}  (fixed — do not change)')
print()
print('  Output files')
print('    filtered_positive.tsv   : clustered positive metadata')
print('    filtered_negative.tsv   : clustered negative metadata')
print('    training_with_folds.tsv : training set with label + fold columns')
print('    benchmarking_set.tsv    : held-out benchmark with label column')
print()
print(sep)